# WGAN-GP sobre Fashion-MNIST

**Dataset real:** fashion_mnist  
**Modelo:** wgan-gp  
**Objetivo:** Estudiar estabilidad generativa y gradient penalty con imágenes reales.

## 1. Pregunta experimental

Comparar DCGAN y WGAN-GP con igual presupuesto de actualizaciones.

In [ ]:
from pathlib import Path
import sys, json
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p/'pyproject.toml').exists())
sys.path.insert(0, str(ROOT/'src'))
TRACK_ID = '28_wgan_gp'
SPLIT_SEED = 42
TRAINING_SEED = 42
QUICK = True

## 2. Procedencia y contrato de datos

Lea el manifiesto antes de descargar. El test permanece sellado hasta elegir el checkpoint.

In [ ]:
import yaml
manifest = yaml.safe_load((ROOT/'advanced_labs'/TRACK_ID/'data'/'dataset.yaml').read_text())
manifest

## 3. Fundamento específico

Distancia Wasserstein, crítico sin sigmoide y restricción Lipschitz mediante gradient penalty.

In [ ]:
from neural_labs.advanced.datasets import load_advanced_data
data = load_advanced_data(TRACK_ID, quick=QUICK, split_seed=SPLIT_SEED)
{'train': len(data.train), 'validation': len(data.validation), 'test': len(data.test), 'classes': data.class_names[:10]}

## 4. Inspección del dominio

Muestras por época, interpolación latente, pérdidas y cobertura de clases mediante clasificador externo.

In [ ]:
image,label=data.train[0]
print(image.shape, float(image.min()), float(image.max()), data.class_names[label])

## 5. Línea base

Antes del modelo avanzado, compare con **DCGAN convolucional**. La línea base se selecciona con validation, nunca con test.

In [ ]:
controls = ['misma partición', 'mismo presupuesto de entrenamiento']
primary_metric = 'wasserstein_estimate'
controls, primary_metric

## 6. Entrenamiento reproducible

El comando guarda configuración, manifiesto, mejor checkpoint, lock experimental e indicadores finales.

In [ ]:
from neural_labs.advanced.training import train_advanced
# Descomente para ejecutar la descarga y el entrenamiento real.
# result = train_advanced(TRACK_ID, quick=QUICK, split_seed=SPLIT_SEED, training_seed=TRAINING_SEED)
# result

## 7. Selección y sellado

Verifique que `experiment.lock.json` se escriba después de seleccionar con validation y antes de calcular test.

In [ ]:
answer = 'Porque cualquier ajuste posterior incorpora información del test y vuelve optimista la estimación de generalización.'
answer

## 8. Evaluación multi-semilla

Repita con `training_seed` diferentes manteniendo fijo `split_seed`; reporte media, desviación y peor caso.

In [ ]:
training_seeds = [41, 42, 43, 44, 45]
# results = [train_advanced(TRACK_ID, quick=True, split_seed=SPLIT_SEED, training_seed=s)['metrics'] for s in training_seeds]

## 9. Análisis de errores y riesgos

Las métricas generativas aproximadas no sustituyen evaluación humana ni validación del uso previsto.

In [ ]:
error_slice = 'clase o condición con menor métrica'
not_recommended = 'decisiones de alto riesgo sin validación externa'
error_slice, not_recommended

## 10. Entregables

1. Notebook ejecutado. 2. Tabla multi-semilla. 3. Análisis de errores. 4. Model card. 5. Evidencia de sellado de test.

In [ ]:
summary = {'track': TRACK_ID, 'split_seed': SPLIT_SEED, 'training_seed': TRAINING_SEED, 'synthetic_data': False}
summary

<!-- ejercicios-evaluables -->
## Ejercicios evaluables

Cinco ejercicios sobre el contrato experimental de esta especialización. Se resuelven con Python estándar: **no hace falta descargar el dataset ni entrenar**, y cada uno trae debajo una celda de comprobación que debe pasar sin error.

### Ejercicio 1 — Auditar la partición

Antes de entrenar hay que demostrar que ningún ejemplo aparece en dos particiones. Una sola fila compartida entre `train` y `test` infla la métrica final sin dar ningún síntoma.

Escribe `sin_solapamiento(train_ids, validation_ids, test_ids)`, que devuelve `True` solo si los tres conjuntos de identificadores son disjuntos dos a dos.

In [ ]:
# SOLUCIÓN DE REFERENCIA
def sin_solapamiento(train_ids, validation_ids, test_ids):
    train, validation, test = set(train_ids), set(validation_ids), set(test_ids)
    return not (train & validation or train & test or validation & test)

In [ ]:
assert sin_solapamiento([1, 2, 3], [4, 5], [6]) is True
assert sin_solapamiento([1, 2, 3], [3, 4], [6]) is False   # solapa train y validation
assert sin_solapamiento([1, 2], [4], [2]) is False         # solapa train y test
assert sin_solapamiento([1], [2, 3], [3]) is False         # solapa validation y test
assert sin_solapamiento([], [], []) is True
print('Auditoría correcta.')

### Ejercicio 2 — Elegir el checkpoint con `validation`

Este laboratorio selecciona con **`wasserstein_estimate`**, donde **menor es mejor**. El modelo que se conserva es el de la época con mejor valor *en validación*, nunca en test.

Escribe `mejor_epoca(historial)`, que recibe una lista de diccionarios con las claves `epoch` y `wasserstein_estimate` y devuelve el número de la mejor época.

In [ ]:
# SOLUCIÓN DE REFERENCIA
SELECTION_METRIC = 'wasserstein_estimate'
HIGHER_IS_BETTER = False

def mejor_epoca(historial):
    elegir = max if HIGHER_IS_BETTER else min
    mejor = elegir(historial, key=lambda fila: fila[SELECTION_METRIC])
    return mejor['epoch']

In [ ]:
historial = [
    {'epoch': 1, 'wasserstein_estimate': 0.40},
    {'epoch': 2, 'wasserstein_estimate': 0.65},
    {'epoch': 3, 'wasserstein_estimate': 0.55},
]
assert HIGHER_IS_BETTER is False
assert mejor_epoca(historial) == 1
print('Regla de selección correcta.')

### Ejercicio 3 — Comparar contra la línea base

La línea base de este laboratorio es **DCGAN convolucional**. Superarla por poco no basta: la diferencia tiene que ser mayor que la dispersión del propio modelo entre semillas, o no se puede distinguir de la suerte.

Escribe `supera_linea_base(puntajes, base)`, que recibe la lista de puntajes del modelo (uno por semilla) y el puntaje de la línea base, y devuelve un diccionario con `media`, `desviacion` y `concluyente`.

In [ ]:
# SOLUCIÓN DE REFERENCIA
from statistics import mean, pstdev

def supera_linea_base(puntajes, base):
    media = mean(puntajes)
    desviacion = pstdev(puntajes) if len(puntajes) > 1 else 0.0
    ventaja = media - base if False else base - media
    return {
        'media': media,
        'desviacion': desviacion,
        'concluyente': ventaja > desviacion,
    }

In [ ]:
claro = supera_linea_base([0.20, 0.21, 0.22], 0.60)
dudoso = supera_linea_base([0.45, 0.55, 0.65], 0.60)
assert round(claro['media'], 4) == 0.21
assert claro['concluyente'] is True
assert dudoso['concluyente'] is False   # la mejora cabe dentro del ruido
print('Comparación correcta:', claro)

### Ejercicio 4 — No abrir `test` sin sellar

El repositorio escribe `experiment.lock.json` con las semillas, la configuración y el checkpoint elegido **antes** de evaluar `test`. Sin ese archivo, cualquier número de test es inválido.

Escribe `puede_abrir_test(run_dir)`, que devuelve `True` solo si el directorio de la ejecución contiene `experiment.lock.json`.

In [ ]:
# SOLUCIÓN DE REFERENCIA
from pathlib import Path

def puede_abrir_test(run_dir):
    return (Path(run_dir) / 'experiment.lock.json').is_file()

In [ ]:
import tempfile
from pathlib import Path

with tempfile.TemporaryDirectory() as carpeta:
    ruta = Path(carpeta)
    assert puede_abrir_test(ruta) is False   # todavía no se selló
    (ruta / 'experiment.lock.json').write_text('{}', encoding='utf-8')
    assert puede_abrir_test(ruta) is True
print('Sellado comprobado.')

### Ejercicio 5 — Dejar el plan por escrito

El experimento propio de esta ruta es: **Comparar DCGAN y WGAN-GP con igual presupuesto de actualizaciones**.

Declara el plan antes de ejecutarlo: qué hipótesis pones a prueba, qué variable cambias, qué mantienes fijo, con qué semillas de entrenamiento y qué conclusión esperas poder escribir. Completa las cinco variables.

In [ ]:
# SOLUCIÓN DE REFERENCIA
HIPOTESIS = 'Comparar DCGAN y WGAN-GP con igual presupuesto de actualizaciones mejora wasserstein_estimate frente a DCGAN convolucional.'
VARIABLE_QUE_CAMBIA = 'Comparar DCGAN y WGAN-GP con igual presupuesto de actualizaciones'
VARIABLES_CONTROLADAS = ['misma partición (split_seed=42)', 'mismo presupuesto de épocas', 'mismo dataset: fashion_mnist']
SEMILLAS_DE_ENTRENAMIENTO = [41, 42, 43]
CONCLUSION = ('La diferencia solo es concluyente si supera la dispersión entre semillas; si no, se reporta que no se distingue del ruido, y se compara siempre contra DCGAN convolucional.')

In [ ]:
assert len(HIPOTESIS) >= 30, 'La hipótesis debe poder resultar falsa; descríbela.'
assert len(VARIABLE_QUE_CAMBIA) >= 10, 'Un experimento cambia una cosa a la vez.'
assert len(VARIABLES_CONTROLADAS) >= 3, 'Enumera al menos tres variables controladas.'
assert len(SEMILLAS_DE_ENTRENAMIENTO) >= 3, 'Con menos de tres semillas no hay dispersión.'
assert len(CONCLUSION) >= 40, 'La conclusión debe hablar de magnitud e incertidumbre.'
print('Plan experimental completo.')